In [1]:
import os
import pyspark

print("Répertoire :", os.getcwd())
print("Version PySpark :", pyspark.__version__)

Répertoire : /home/jovyan/work
Version PySpark : 3.5.3


# TP : Mise en oeuvre d’un pipeline ETL et d’un Data Lakehouse (Bronze/Silver/Gold)

## Preparation de l'envirronnement

### 1.1 Initialisation de la session Spark

In [2]:

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Eshop-ETL-Lakehouse")
    .master("local[*]")
    .getOrCreate()
)

print("Spark version :", spark.version)
print("Spark master :", spark.sparkContext.master)
print("Application ID :", spark.sparkContext.applicationId)


Spark version : 3.5.3
Spark master : local[*]
Application ID : local-1794175898188


### 1.2 Ingestion des données sources

In [3]:
customers_raw =(
    spark.read.format("csv")
    .option("header", "true")
    .option("delimiter", ";")
    .option("inferSchema", "false")
    .load("data/source/customers.csv")
)

customers_raw.printSchema()
customers_raw.show(10, truncate=False)

print("Nombre de clients :", customers_raw.count())

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- registration_date: string (nullable = true)

+-----------+-------------+------+-----------------+
|customer_id|customer_name|region|registration_date|
+-----------+-------------+------+-----------------+
|C001       |Alice 01     |Nord  |2024-01-08       |
|C002       |Youssef 02   |Sud   |2024-01-15       |
|C003       |Sara 03      |Centre|2024-01-22       |
|C004       |Omar 04      |Est   |2024-01-29       |
|C005       |Nadia 05     |Ouest |2024-02-05       |
|C006       |Amine 06     |Nord  |2024-02-12       |
|C007       |Alice 07     |Sud   |2024-02-19       |
|C008       |Youssef 08   |Centre|2024-02-26       |
|C009       |Sara 09      |Est   |2024-03-04       |
|C010       |Omar 10      |Ouest |2024-03-11       |
+-----------+-------------+------+-----------------+
only showing top 10 rows

Nombre de clients : 32


In [4]:
# Let's continue the ingestion process by reading the other data sources
def read_csv_data(filename):
    return (
        spark.read.format("csv")
        .option("header", "true")
        .option("delimiter", ";")
        .option("inferSchema", "false")
        .csv(f"data/source/{filename}")
    )

products_raw = read_csv_data("products.csv")
orders_raw = read_csv_data("orders.csv")
order_items_raw = read_csv_data("order_items.csv")


In [5]:
#promotions file
promotions_raw = spark.read.text("data/source/promotions.jsonl")
promotions_raw.printSchema()
promotions_raw.show(truncate=False)

root
 |-- value: string (nullable = true)

+----------------------------------------------------------------------------------------------------+
|value                                                                                               |
+----------------------------------------------------------------------------------------------------+
|{"promotion_code":"PROMO5","discount_rate":0.05,"start_date":"2025-01-01","end_date":"2025-12-31"}  |
|{"promotion_code":"PROMO10","discount_rate":0.1,"start_date":"2025-05-01","end_date":"2025-05-31"}  |
|{"promotion_code":"SUMMER15","discount_rate":0.15,"start_date":"2025-06-01","end_date":"2025-08-31"}|
|{"promotion_code":"VIP20","discount_rate":0.2,"start_date":"2025-01-01","end_date":"2025-12-31"}    |
|{"promotion_code":"OLD30","discount_rate":0.3,"start_date":"2024-01-01","end_date":"2024-12-31"}    |
|{"promotion_code":"BAD_JSON","discount_rate":0.10,"start_date":"2025-06-01","end_date":             |
+-----------------------------

In [6]:
# number of lines in each source
sources = {
    "customers": customers_raw,
    "products": products_raw,
    "orders": orders_raw,
    "order_items": order_items_raw,
    "promotions": promotions_raw
}

for name, df in sources.items():
    print(f"{name}: {df.count()} lignes")


customers: 32 lignes
products: 14 lignes
orders: 44 lignes
order_items: 86 lignes
promotions: 6 lignes


### Data Exploration

In [7]:
for name, df in sources.items():
    print(f"\n===== {name.upper()} =====")

    df.printSchema()
    df.show(5, truncate=False)


===== CUSTOMERS =====
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- registration_date: string (nullable = true)

+-----------+-------------+------+-----------------+
|customer_id|customer_name|region|registration_date|
+-----------+-------------+------+-----------------+
|C001       |Alice 01     |Nord  |2024-01-08       |
|C002       |Youssef 02   |Sud   |2024-01-15       |
|C003       |Sara 03      |Centre|2024-01-22       |
|C004       |Omar 04      |Est   |2024-01-29       |
|C005       |Nadia 05     |Ouest |2024-02-05       |
+-----------+-------------+------+-----------------+
only showing top 5 rows


===== PRODUCTS =====
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- unit_price: string (nullable = true)
 |-- unit_cost: string (nullable = true)

+----------+------------+------------+----------+--------

In [8]:
# Volumes verification

expected_counts = {
    "customers": 32,
    "products": 14,
    "orders": 44,
    "order_items": 86,
    "promotions": 6
}

for name, df in sources.items():
    actual = df.count()
    expected = expected_counts[name]

    print(
        f"{name}: attendu={expected}, "
        f"obtenu={actual}"
    )

    assert actual == expected, (
        f"Volume incorrect pour {name}"
    )

customers: attendu=32, obtenu=32
products: attendu=14, obtenu=14
orders: attendu=44, obtenu=44
order_items: attendu=86, obtenu=86
promotions: attendu=6, obtenu=6



### Conclusion de l'ingestion

Les cinq sources ont été chargées dans des DataFrames Spark.

Les champs CSV ont été conservés sous forme de chaînes
de caractères afin de préserver leurs valeurs originales.

Le fichier JSONL a été lu en mode texte pour conserver
également les lignes mal formées.

Les volumes ont été comparés aux valeurs de référence.

Les contrôles de qualité, les conversions de types et la
gestion des anomalies seront réalisés dans la couche Silver.


# Bronze Zone

In [9]:
# Original files consservation 

from pathlib import Path
import shutil

project_dir = Path("/home/jovyan/work")

source_dir = project_dir / "data" / "source"
bronze_dir = project_dir / "data_lakehouse" / "bronze" / "original"

bronze_dir.mkdir(parents=True, exist_ok=True)

filenames = [
    "customers.csv",
    "products.csv",
    "orders.csv",
    "order_items.csv",
    "promotions.jsonl"
]

for filename in filenames:
    source_file = source_dir / filename
    destination_file = bronze_dir / filename

    shutil.copy2(source_file, destination_file)

    print(f"Copié : {filename}")


Copié : customers.csv
Copié : products.csv
Copié : orders.csv
Copié : order_items.csv
Copié : promotions.jsonl


In [10]:
# Copy intgrity verification

import hashlib

def file_hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

for filename in filenames:
    original = source_dir / filename
    bronze = bronze_dir / filename

    assert file_hash(original) == file_hash(bronze)

    print(f"{filename}: copie identique")

customers.csv: copie identique
products.csv: copie identique
orders.csv: copie identique
order_items.csv: copie identique
promotions.jsonl: copie identique


#### Json manifest

In [11]:
from uuid import uuid4
from datetime import datetime, timezone

run_id = str(uuid4())

print("Run ID :", run_id)
print("Date :", datetime.now(timezone.utc).isoformat())

Run ID : 0e74c7d6-9662-497e-9656-b82e552fcf7d
Date : 2026-11-08T22:11:49.858062+00:00


In [12]:
import json
import shutil

manifest = []

for filename in filenames:
    source_path = source_dir / filename
    bronze_path = bronze_dir / filename

    record = {
        "run_id": run_id,
        "source": filename,
        "ingestion_time": datetime.now(timezone.utc).isoformat(),
        "bronze_path": str(bronze_path),
        "row_count": None,
        "status": "failed"
    }

    try:
        shutil.copy2(source_path, bronze_path)

        assert file_hash(source_path) == file_hash(bronze_path)

        if filename.endswith(".jsonl"):
            df = spark.read.text(str(bronze_path))
        else:
            df = (
                spark.read
                .option("header", True)
                .option("sep", ";")
                .option("inferSchema", False)
                .csv(str(bronze_path))
            )

        record["row_count"] = df.count()
        record["sha256"] = file_hash(bronze_path)
        record["status"] = "success"

    except Exception as e:
        record["error"] = str(e)

    manifest.append(record)

    print(filename, record["status"], record["row_count"])


customers.csv success 32
products.csv success 14
orders.csv success 44
order_items.csv success 86
promotions.jsonl success 6


In [13]:
metadata_dir = project_dir / "data_lakehouse" / "metadata" / "ingestion"

metadata_dir.mkdir(parents=True, exist_ok=True)

manifest_path = metadata_dir / f"manifest_{run_id}.json"

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print("Manifeste enregistré :", manifest_path)

assert all(record["status"] == "success" for record in manifest)

Manifeste enregistré : /home/jovyan/work/data_lakehouse/metadata/ingestion/manifest_0e74c7d6-9662-497e-9656-b82e552fcf7d.json


##### Manifest verification

In [14]:
from pathlib import Path
import json

metadata_dir = Path(
    "/home/jovyan/work/data_lakehouse/metadata/ingestion"
)

manifests = sorted(
    metadata_dir.glob("manifest_*.json"),
    key=lambda p: p.stat().st_mtime
)

if manifests:
    latest_manifest = manifests[-1]

    with open(latest_manifest, "r", encoding="utf-8") as f:
        records = json.load(f)

    for record in records:
        print(
            record["source"],
            record["row_count"],
            record["status"],
            record["run_id"]
        )
else:
    print("Aucun manifeste trouvé.")

customers.csv 32 success 0e74c7d6-9662-497e-9656-b82e552fcf7d
products.csv 14 success 0e74c7d6-9662-497e-9656-b82e552fcf7d
orders.csv 44 success 0e74c7d6-9662-497e-9656-b82e552fcf7d
order_items.csv 86 success 0e74c7d6-9662-497e-9656-b82e552fcf7d
promotions.jsonl 6 success 0e74c7d6-9662-497e-9656-b82e552fcf7d


###  we can read with Spark verification

In [15]:
# test in customers file
customers_bronze = (
    spark.read
    .option("header", True)
    .option("sep", ";")
    .option("inferSchema", False)
    .csv(str(bronze_dir / "customers.csv"))
)

customers_count = customers_bronze.count()

print("Nombre de lignes Bronze :", customers_count)

assert customers_count == 32

Nombre de lignes Bronze : 32


In [16]:
# other files 
for record in manifest:

    filename = record["source"]
    path = record["bronze_path"]
    expected_count = record["row_count"]

    if filename.endswith(".jsonl"):
        df = spark.read.text(path)
    else:
        df = (
            spark.read
            .option("header", True)
            .option("sep", ";")
            .option("inferSchema", False)
            .csv(path)
        )

    actual_count = df.count()

    assert actual_count == expected_count, (
        f"Volume incorrect pour {filename}"
    )

    print(f"{filename} : {actual_count} lignes  OK")

customers.csv : 32 lignes  OK
products.csv : 14 lignes  OK
orders.csv : 44 lignes  OK
order_items.csv : 86 lignes  OK
promotions.jsonl : 6 lignes  OK


## Silver

In [17]:
# Idemtiffy required columns for cusomers
from pyspark.sql import functions as F

required_columns = [
    "customer_id",
    "customer_name",
    "region"
]

for column in required_columns:
    missing_count = customers_bronze.filter(
        F.col(column).isNull()
        | (F.trim(F.col(column)) == "")
    ).count()

    print(f"{column}: {missing_count} valeurs manquantes")

customer_id: 0 valeurs manquantes
customer_name: 1 valeurs manquantes
region: 0 valeurs manquantes


In [18]:
# Invalid dates

customers_typed = customers_bronze.withColumn(
    "registration_date_parsed",
    F.expr("try_cast(registration_date as date)")
)

customers_typed.select(
    "customer_id",
    "registration_date",
    "registration_date_parsed"
).show(truncate=False)

+-----------+-----------------+------------------------+
|customer_id|registration_date|registration_date_parsed|
+-----------+-----------------+------------------------+
|C001       |2024-01-08       |2024-01-08              |
|C002       |2024-01-15       |2024-01-15              |
|C003       |2024-01-22       |2024-01-22              |
|C004       |2024-01-29       |2024-01-29              |
|C005       |2024-02-05       |2024-02-05              |
|C006       |2024-02-12       |2024-02-12              |
|C007       |2024-02-19       |2024-02-19              |
|C008       |2024-02-26       |2024-02-26              |
|C009       |2024-03-04       |2024-03-04              |
|C010       |2024-03-11       |2024-03-11              |
|C011       |2024-03-18       |2024-03-18              |
|C012       |2024-03-25       |2024-03-25              |
|C013       |2024-04-01       |2024-04-01              |
|C014       |2024-04-08       |2024-04-08              |
|C015       |2024-04-15       |

In [19]:
customers_typed.filter(
    F.col("registration_date").isNotNull()
    & F.col("registration_date_parsed").isNull()
).select(
    "customer_id",
    "registration_date",
    "registration_date_parsed"
).show(truncate=False)

+-----------+-----------------+------------------------+
|customer_id|registration_date|registration_date_parsed|
+-----------+-----------------+------------------------+
|C031       |2025-02-30       |NULL                    |
+-----------+-----------------+------------------------+



In [20]:
# Detect duplicates id in customers file 
duplicate_ids = (
    customers_bronze.groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)
duplicate_ids.show()

+-----------+-----+
|customer_id|count|
+-----------+-----+
|       C005|    2|
+-----------+-----+



In [21]:
# Detect exact duplicates in customers file ( in all columns)
exact_duplicates = (
    customers_bronze
    .groupBy(*customers_bronze.columns)
    .count()
    .filter(F.col("count") > 1)
)

exact_duplicates.show(truncate=False)


+-----------+-------------+------+-----------------+-----+
|customer_id|customer_name|region|registration_date|count|
+-----------+-------------+------+-----------------+-----+
|C005       |Nadia 05     |Ouest |2024-02-05       |2    |
+-----------+-------------+------+-----------------+-----+



In [22]:

# Build quality reasons for customers

customers_checked = customers_typed.withColumn(
    "quality_reason",
    F.concat_ws(
        " | ",
        F.when(
            F.col("customer_id").isNull() |
            (F.trim(F.col("customer_id")) == ""),
            "ID_MANQUANT"
        ),
        F.when(
            F.col("customer_name").isNull() |
            (F.trim(F.col("customer_name")) == ""),
            "NOM_MANQUANT"
        ),
        F.when(
            F.col("region").isNull() |
            (F.trim(F.col("region")) == ""),
            "REGION_MANQUANTE"
        ),
        F.when(
            F.col("registration_date_parsed").isNull(),
            "DATE_INVALIDE"
        )
    )
)

In [23]:
# verification of quality reasons
customers_checked.filter(
    F.col("quality_reason") != ""
).select(
    "customer_id",
    "customer_name",
    "registration_date",
    "quality_reason"
).show(truncate=False)

+-----------+-------------+-----------------+----------------------------+
|customer_id|customer_name|registration_date|quality_reason              |
+-----------+-------------+-----------------+----------------------------+
|C031       |NULL         |2025-02-30       |NOM_MANQUANT | DATE_INVALIDE|
+-----------+-------------+-----------------+----------------------------+



In [24]:
# Mange duplicates in customers 
customers_unique = customers_checked.dropDuplicates(
    customers_bronze.columns
)

print("Avant :", customers_checked.count())
print("Après :", customers_unique.count())

Avant : 32
Après : 31


##### Duplicates audit

In [25]:
customers_duplicates_audit = exact_duplicates.withColumn(
    "extra_copies",
    F.col("count") - 1
)

customers_duplicates_audit.show(truncate=False)

+-----------+-------------+------+-----------------+-----+------------+
|customer_id|customer_name|region|registration_date|count|extra_copies|
+-----------+-------------+------+-----------------+-----+------------+
|C005       |Nadia 05     |Ouest |2024-02-05       |2    |1           |
+-----------+-------------+------+-----------------+-----+------------+



In [26]:
# id conflicts in customers file
customers_conflicts = (
    customers_unique
    .filter(
        F.col("customer_id").isNotNull()
        & (F.trim(F.col("customer_id")) != "")
    )
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

customers_conflicts.show(truncate=False)

+-----------+-----+
|customer_id|count|
+-----------+-----+
+-----------+-----+



In [27]:
# Moove invalid customers to quarantine
customers_valid = customers_unique.filter(
    F.col("quality_reason") == ""
)

customers_quarantine = customers_unique.filter(
    F.col("quality_reason") != ""
)

print("Valid customers:", customers_valid.count())
print("Quarantined customers:", customers_quarantine.count())

Valid customers: 30
Quarantined customers: 1


In [28]:
# verification of the quarantine
customers_quarantine.select(
    "customer_id",
    "customer_name",
    "registration_date",
    "quality_reason"
).show(truncate=False)

+-----------+-------------+-----------------+----------------------------+
|customer_id|customer_name|registration_date|quality_reason              |
+-----------+-------------+-----------------+----------------------------+
|C031       |NULL         |2025-02-30       |NOM_MANQUANT | DATE_INVALIDE|
+-----------+-------------+-----------------+----------------------------+



In [29]:
# Data conservation verification
bronze_count = customers_bronze.count()
valid_count = customers_valid.count()
quarantine_count = customers_quarantine.count()

audit_count = (
    customers_duplicates_audit
    .agg(
        F.coalesce(
            F.sum("extra_copies"),
            F.lit(0)
        ).alias("total")
    )
    .first()["total"]
)

total = valid_count + quarantine_count + audit_count

print("Bronze:", bronze_count)
print("Valid:", valid_count)
print("Quarantine:", quarantine_count)
print("Duplicates audit:", audit_count)
print("Total:", total)

assert bronze_count == total

print("Data conservation verified")

Bronze: 32
Valid: 30
Quarantine: 1
Duplicates audit: 1
Total: 32
Data conservation verified


### Save the data 

In [30]:

# Define storage paths

silver_path = project_dir / "data_lakehouse" / "silver" / "customers"

quarantine_path = project_dir / "data_lakehouse" / "quarantine" / "customers"

audit_path = project_dir / "data_lakehouse" / "metadata" / "duplicates" / "customers"

# Save results as Parquet

customers_valid.write.mode("overwrite").parquet(str(silver_path))

customers_quarantine.write.mode("overwrite").parquet(str(quarantine_path))

customers_duplicates_audit.write.mode("overwrite").parquet(str(audit_path))

print("Customers data successfully persisted ")


Customers data successfully persisted 


In [31]:
# Final verification of persisted data
customers_silver_reloaded = spark.read.parquet(
    str(silver_path)
)

customers_quarantine_reloaded = spark.read.parquet(
    str(quarantine_path)
)

customers_audit_reloaded = spark.read.parquet(
    str(audit_path)
)

print("Silver:", customers_silver_reloaded.count())
print("Quarantine:", customers_quarantine_reloaded.count())
print("Audit groups:", customers_audit_reloaded.count())

customers_silver_reloaded.printSchema()

Silver: 30
Quarantine: 1
Audit groups: 1
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- registration_date: string (nullable = true)
 |-- registration_date_parsed: date (nullable = true)
 |-- quality_reason: string (nullable = true)



#### automate rest of the silver pipeline

In [32]:
from pyspark.sql import functions as F, types as T

def missing(column):
    return F.col(column).isNull() | (F.trim(F.col(column)) == "")

def add_reasons(df, checks):
    return df.withColumn(
        "quality_reason",
        F.concat_ws(" | ", *[
            F.when(condition, F.lit(label))
            for label, condition in checks
        ])
    )

def read_bronze(name):
    return (
        spark.read
        .option("header", True)
        .option("sep", ";")
        .option("inferSchema", False)
        .csv(str(bronze_dir / f"{name}.csv"))
    )

def save_quality(name, df, key, raw_cols):
    # Audit des doublons exacts
    audit = (
        df.groupBy(*raw_cols).count()
        .filter(F.col("count") > 1)
        .withColumn("extra_copies", F.col("count") - 1)
    )

    unique = df.dropDuplicates(raw_cols)

    # Conflits d'identifiants
    conflicts = (
        unique.filter(~missing(key))
        .groupBy(key).count()
        .filter(F.col("count") > 1)
        .select(key)
        .withColumn("is_conflict", F.lit(True))
    )

    checked = (
        unique.join(conflicts, on=key, how="left")
        .withColumn(
            "quality_reason",
            F.concat_ws(
                " | ",
                F.col("quality_reason"),
                F.when(F.col("is_conflict") == True,
                       "CONFLIT_IDENTIFIANT")
            )
        )
        .drop("is_conflict")
    )

    valid = checked.filter(F.col("quality_reason") == "")
    rejected = checked.filter(F.col("quality_reason") != "")

    root = project_dir / "data_lakehouse"

    valid.write.mode("overwrite").parquet(
        str(root / "silver" / name)
    )
    rejected.write.mode("overwrite").parquet(
        str(root / "quarantine" / name)
    )
    audit.write.mode("overwrite").parquet(
        str(root / "metadata" / "duplicates" / name)
    )

    copies = audit.agg(
        F.coalesce(F.sum("extra_copies"), F.lit(0)).alias("n")
    ).first()["n"]

    raw_n = df.count()
    valid_n = valid.count()
    rejected_n = rejected.count()

    assert raw_n == valid_n + rejected_n + copies

    print(
        f"{name}: {raw_n} = {valid_n} Silver + "
        f"{rejected_n} rejets + {copies} copies"
    )

    return valid, rejected, audit

##### products file rules 

In [33]:
p = read_bronze("products")

p = (
    p.withColumn(
        "unit_price_parsed",
        F.expr("try_cast(unit_price as decimal(12,2))")
    )
    .withColumn(
        "unit_cost_parsed",
        F.expr("try_cast(unit_cost as decimal(12,2))")
    )
)

p = add_reasons(p, [
    ("ID_MANQUANT", missing("product_id")),
    ("NOM_MANQUANT", missing("product_name")),
    ("CATEGORIE_MANQUANTE", missing("category")),
    ("PRIX_INVALIDE",
     F.col("unit_price_parsed").isNull() |
     (F.col("unit_price_parsed") < 0)),
    ("COUT_INVALIDE",
     F.col("unit_cost_parsed").isNull() |
     (F.col("unit_cost_parsed") < 0))
])

products_silver, products_quarantine, products_audit = save_quality(
    "products", p, "product_id", read_bronze("products").columns
)

products: 14 = 12 Silver + 1 rejets + 1 copies


In [34]:
# same for promotions 
promo_schema = T.StructType([
    T.StructField("promotion_code", T.StringType()),
    T.StructField("discount_rate", T.StringType()),
    T.StructField("start_date", T.StringType()),
    T.StructField("end_date", T.StringType())
])

pr = spark.read.text(str(bronze_dir / "promotions.jsonl"))

pr = (
    pr.withColumn("parsed", F.from_json("value", promo_schema))
    .withColumn("json_ok", F.col("parsed").isNotNull())
    .select("value", "json_ok", "parsed.*")
    .withColumn(
        "discount_rate_parsed",
        F.expr("try_cast(discount_rate as decimal(6,4))")
    )
    .withColumn(
        "start_date_parsed",
        F.expr("try_cast(start_date as date)")
    )
    .withColumn(
        "end_date_parsed",
        F.expr("try_cast(end_date as date)")
    )
)

pr = add_reasons(pr, [
    ("JSON_MALFORME", ~F.col("json_ok")),
    ("CODE_MANQUANT", missing("promotion_code")),
    ("TAUX_INVALIDE",
     F.col("discount_rate_parsed").isNull() |
     (F.col("discount_rate_parsed") < 0) |
     (F.col("discount_rate_parsed") > 1)),
    ("DATE_DEBUT_INVALIDE", F.col("start_date_parsed").isNull()),
    ("DATE_FIN_INVALIDE", F.col("end_date_parsed").isNull()),
    ("PERIODE_INCOHERENTE",
     F.col("start_date_parsed") > F.col("end_date_parsed"))
])

promotions_silver, promotions_quarantine, promotions_audit = save_quality(
    "promotions", pr, "promotion_code", ["value"]
)

promotions: 6 = 5 Silver + 1 rejets + 0 copies


In [35]:
# orders 

orders = read_bronze("orders")

orders = (
    orders.withColumn(
        "order_date_parsed",
        F.expr("try_cast(order_date as date)")
    )
    .join(
        customers_valid.select("customer_id")
        .withColumn("client_found", F.lit(True)),
        on="customer_id",
        how="left"
    )
)
orders = add_reasons(orders, [
    ("ID_MANQUANT", missing("order_id")),
    ("CLIENT_MANQUANT", missing("customer_id")),
    ("CLIENT_INCONNU",
     ~missing("customer_id") & F.col("client_found").isNull()),
    ("DATE_INVALIDE", F.col("order_date_parsed").isNull()),
    ("STATUT_INVALIDE",
     missing("status") |
     ~F.col("status").isin(
         "PAID", "DELIVERED", "PENDING", "CANCELLED"
     ))
])

ref_promos = promotions_silver.select(
    F.col("promotion_code").alias("ref_code"),
    F.col("start_date_parsed").alias("promo_start"),
    F.col("end_date_parsed").alias("promo_end")
)

orders = (
    orders.join(
        ref_promos,
        orders.promotion_code == F.col("ref_code"),
        "left"
    )
    .withColumn(
        "promotion_issue",
        F.when(missing("promotion_code"), "")
        .when(F.col("ref_code").isNull(), "PROMO_INCONNUE")
        .when(
            ~F.col("order_date_parsed").between(
                F.col("promo_start"), F.col("promo_end")
            ),
            "PROMO_HORS_VALIDITE"
        )
        .otherwise("")
    )
)

orders_silver, orders_quarantine, orders_audit = save_quality(
    "orders", orders, "order_id", read_bronze("orders").columns
)


orders: 44 = 40 Silver + 3 rejets + 1 copies


In [36]:
#order_items
items = read_bronze("order_items")

items = (
    items.withColumn(
        "quantity_parsed",
        F.expr("try_cast(quantity as int)")
    )
    .withColumn(
        "unit_price_parsed",
        F.expr("try_cast(unit_price as decimal(12,2))")
    )
    .join(
        orders_silver.select("order_id")
        .withColumn("order_found", F.lit(True)),
        on="order_id",
        how="left"
    )
    .join(
        products_silver.select("product_id")
        .withColumn("product_found", F.lit(True)),
        on="product_id",
        how="left"
    )
)

items = add_reasons(items, [
    ("ID_MANQUANT", missing("order_item_id")),
    ("COMMANDE_INCONNUE",
     missing("order_id") | F.col("order_found").isNull()),
    ("PRODUIT_INCONNU",
     missing("product_id") | F.col("product_found").isNull()),
    ("QUANTITE_INVALIDE",
     F.col("quantity_parsed").isNull() |
     (F.col("quantity_parsed") <= 0)),
    ("PRIX_INVALIDE",
     F.col("unit_price_parsed").isNull() |
     (F.col("unit_price_parsed") < 0))
])

items_silver, items_quarantine, items_audit = save_quality(
    "order_items", items, "order_item_id",
    read_bronze("order_items").columns
)

order_items: 86 = 80 Silver + 5 rejets + 1 copies


# Joins and commercial views

In [37]:
references = [
    (orders_silver, "order_id"),
    (products_silver, "product_id"),
    (promotions_silver, "promotion_code")
]

for df, key in references:
    total = df.count()
    distinct_keys = df.select(key).distinct().count()

    assert total == distinct_keys, f"Doublons sur {key}"

    print(f"{key}: clé unique OK")

order_id: clé unique OK
product_id: clé unique OK
promotion_code: clé unique OK


In [38]:
# 
commercial_view = items_silver.select(
    "order_item_id",
    "order_id",
    "product_id",
    "quantity_parsed",
    "unit_price_parsed"
).join(
    orders_silver.select(
        "order_id",
        "customer_id",
        "order_date_parsed",
        "status",
        "promotion_code"
    ),
    on="order_id",
    how="inner"
)

print("Lignes apres jointure :", commercial_view.count())

Lignes apres jointure : 80


In [39]:
# ajout des informations sur les produits
commercial_view = commercial_view.join(
    products_silver.select(
        "product_id",
        "product_name",
        "category",
        F.col("unit_cost_parsed").alias("unit_cost"),
        F.col("unit_price_parsed").alias("catalog_price")
    ),
    on="product_id",
    how="inner"
)

print("Apres jointure produits :", commercial_view.count())

Apres jointure produits : 80


In [40]:
# add promotions
commercial_view = commercial_view.join(
    promotions_silver.select(
        "promotion_code",
        F.col("discount_rate_parsed").alias("discount_rate"),
        F.col("start_date_parsed").alias("promo_start"),
        F.col("end_date_parsed").alias("promo_end")
    ),
    on="promotion_code",
    how="left"
)

print("Apres jointure promotions :", commercial_view.count())


Apres jointure promotions : 80


In [41]:
# commercial view verification
commercial_view.select(
    "order_item_id",
    "order_id",
    "product_name",
    "quantity_parsed",
    "unit_price_parsed",
    "unit_cost",
    "status",
    "promotion_code",
    "discount_rate"
).show(10, truncate=False)

+-------------+--------+------------+---------------+-----------------+---------+---------+--------------+-------------+
|order_item_id|order_id|product_name|quantity_parsed|unit_price_parsed|unit_cost|status   |promotion_code|discount_rate|
+-------------+--------+------------+---------------+-----------------+---------+---------+--------------+-------------+
|I0006        |O0003   |Mouse       |1              |20.00            |8.00     |CANCELLED|PROMO10       |0.1000       |
|I0007        |O0004   |USB Hub     |1              |25.00            |11.00    |PENDING  |SUMMER15      |0.1500       |
|I0014        |O0007   |Mouse       |1              |20.00            |8.00     |CANCELLED|OLD30         |0.3000       |
|I0015        |O0008   |USB Hub     |1              |25.00            |11.00    |PENDING  |NULL          |NULL         |
|I0022        |O0011   |Mouse       |1              |20.00            |8.00     |CANCELLED|SUMMER15      |0.1500       |
|I0023        |O0012   |USB Hub 

In [42]:
# commercial view final verification
commercial_view.select(
    "order_item_id",
    "order_id",
    "product_name",
    "quantity_parsed",
    "unit_price_parsed",
    "unit_cost",
    "status",
    "promotion_code",
    "discount_rate"
).show(10, truncate=False)

+-------------+--------+------------+---------------+-----------------+---------+---------+--------------+-------------+
|order_item_id|order_id|product_name|quantity_parsed|unit_price_parsed|unit_cost|status   |promotion_code|discount_rate|
+-------------+--------+------------+---------------+-----------------+---------+---------+--------------+-------------+
|I0006        |O0003   |Mouse       |1              |20.00            |8.00     |CANCELLED|PROMO10       |0.1000       |
|I0007        |O0004   |USB Hub     |1              |25.00            |11.00    |PENDING  |SUMMER15      |0.1500       |
|I0014        |O0007   |Mouse       |1              |20.00            |8.00     |CANCELLED|OLD30         |0.3000       |
|I0015        |O0008   |USB Hub     |1              |25.00            |11.00    |PENDING  |NULL          |NULL         |
|I0022        |O0011   |Mouse       |1              |20.00            |8.00     |CANCELLED|SUMMER15      |0.1500       |
|I0023        |O0012   |USB Hub 

In [43]:
# validite des promotions
commercial_view = commercial_view.withColumn(
    "promotion_status",
    F.when(
        F.col("promotion_code").isNull() |
        (F.trim(F.col("promotion_code")) == ""),
        "AUCUNE_PROMOTION"
    )
    .when(
        F.col("discount_rate").isNull(),
        "PROMO_INCONNUE"
    )
    .when(
        F.col("order_date_parsed").between(
            F.col("promo_start"),
            F.col("promo_end")
        ),
        "APPLICABLE"
    )
    .otherwise("HORS_VALIDITE")
)

In [44]:
commercial_view = commercial_view.withColumn(
    "applicable_discount_rate",
    F.when(
        F.col("promotion_status") == "APPLICABLE",
        F.col("discount_rate")
    ).otherwise(F.lit(0).cast("decimal(6,4)"))
)

In [45]:
# results verification

commercial_view.select(
    "order_id",
    "order_date_parsed",
    "promotion_code",
    "discount_rate",
    "promotion_status",
    "applicable_discount_rate"
).orderBy("order_id").show(20, truncate=False)

+--------+-----------------+--------------+-------------+----------------+------------------------+
|order_id|order_date_parsed|promotion_code|discount_rate|promotion_status|applicable_discount_rate|
+--------+-----------------+--------------+-------------+----------------+------------------------+
|O0001   |2025-05-04       |NULL          |NULL         |AUCUNE_PROMOTION|0.0000                  |
|O0001   |2025-05-04       |NULL          |NULL         |AUCUNE_PROMOTION|0.0000                  |
|O0002   |2025-05-07       |PROMO5        |0.0500       |APPLICABLE      |0.0500                  |
|O0002   |2025-05-07       |PROMO5        |0.0500       |APPLICABLE      |0.0500                  |
|O0003   |2025-05-10       |PROMO10       |0.1000       |APPLICABLE      |0.1000                  |
|O0003   |2025-05-10       |PROMO10       |0.1000       |APPLICABLE      |0.1000                  |
|O0004   |2025-05-13       |SUMMER15      |0.1500       |HORS_VALIDITE   |0.0000                  |


# Gold

In [46]:

# Select eligible sales

gold_lines = commercial_view.filter(
    F.col("status").isin("PAID", "DELIVERED")
)

print("Gold lines:", gold_lines.count())

print(
    "Eligible orders:",
    gold_lines.select("order_id").distinct().count()
)

Gold lines: 40
Eligible orders: 20


In [47]:
# Compute financial metrics per order item

gold_lines = (
    gold_lines
    .withColumn(
        "net_revenue",
        F.round(
            F.col("quantity_parsed")
            * F.col("unit_price_parsed")
            * (F.lit(1) - F.col("applicable_discount_rate")),
            2
        ).cast("decimal(18,2)")
    )
    .withColumn(
        "total_cost",
        F.round(
            F.col("quantity_parsed") * F.col("unit_cost"),
            2
        ).cast("decimal(18,2)")
    )
    .withColumn(
        "margin",
        F.col("net_revenue") - F.col("total_cost")
    )
)

In [48]:
## display the results
gold_lines.select(
    "order_id",
    "product_name",
    "quantity_parsed",
    "unit_price_parsed",
    "applicable_discount_rate",
    "net_revenue",
    "total_cost",
    "margin"
).orderBy("order_id").show(10, truncate=False)

+--------+------------+---------------+-----------------+------------------------+-----------+----------+------+
|order_id|product_name|quantity_parsed|unit_price_parsed|applicable_discount_rate|net_revenue|total_cost|margin|
+--------+------------+---------------+-----------------+------------------------+-----------+----------+------+
|O0001   |Screen      |2              |200.00           |0.0000                  |400.00     |280.00    |120.00|
|O0001   |Printer     |3              |180.00           |0.0000                  |540.00     |375.00    |165.00|
|O0002   |Headphones  |3              |75.00            |0.0500                  |213.75     |120.00    |93.75 |
|O0002   |Microphone  |4              |65.00            |0.0500                  |247.00     |140.00    |107.00|
|O0005   |Printer     |3              |180.00           |0.2000                  |432.00     |375.00    |57.00 |
|O0005   |Screen      |2              |190.00           |0.2000                  |304.00     |28

In [ ]:
# verification of the results
assert gold_lines.count() == 40

assert gold_lines.select("order_id").distinct().count() == 20

assert gold_lines.filter(
    F.col("net_revenue").isNull()
    | F.col("total_cost").isNull()
    | F.col("margin").isNull()
).count() == 0

print("Gold validation: OK")

Gold validation: OK


Global Business KPIs

In [51]:
# Calculate global KPIs

global_kpis = gold_lines.agg(
    F.countDistinct("order_id").alias("eligible_orders"),
    F.count("order_item_id").alias("gold_lines_count"),
    F.countDistinct("customer_id").alias("total_customers"),
    F.sum("quantity_parsed").alias("total_quantity"),
    F.sum("net_revenue").alias("net_revenue_total"),
    F.sum("total_cost").alias("total_cost_total"),
    F.sum("margin").alias("total_margin")
)

global_kpis.show(truncate=False)

+---------------+----------------+---------------+--------------+-----------------+----------------+------------+
|eligible_orders|gold_lines_count|total_customers|total_quantity|net_revenue_total|total_cost_total|total_margin|
+---------------+----------------+---------------+--------------+-----------------+----------------+------------+
|20             |40              |20             |120           |13259.80         |9150.00         |4109.80     |
+---------------+----------------+---------------+--------------+-----------------+----------------+------------+



In [54]:
global_kpis = (
    global_kpis
    .withColumn(
        "margin_rate_pct",
        F.when(
            F.col("net_revenue_total") != 0,
            F.round(
                F.col("total_margin")
                / F.col("net_revenue_total") * 100,
                2
            )
        )
    )
    .withColumn(
        "average_order_value",
        F.when(
            F.col("eligible_orders") > 0,
            F.round(
                F.col("net_revenue_total")
                / F.col("eligible_orders"),
                2
            )
        )
    )
)

global_kpis.show(vertical=True, truncate=False)

-RECORD 0-----------------------
 eligible_orders     | 20       
 gold_lines_count    | 40       
 total_customers     | 20       
 total_quantity      | 120      
 net_revenue_total   | 13259.80 
 total_cost_total    | 9150.00  
 total_margin        | 4109.80  
 margin_rate_pct     | 30.99    
 average_order_value | 662.99   



 Analyse des ventes par categorie

In [55]:
# Calculate business KPIs by category
category_kpis = (
    gold_lines
    .groupBy("category")
    .agg(
        F.sum("quantity_parsed").alias("total_quantity"),
        F.count("order_item_id").alias("total_lines"),
        F.sum("net_revenue").alias("net_revenue"),
        F.sum("total_cost").alias("total_cost"),
        F.sum("margin").alias("total_margin")
    )
)
category_kpis.show(truncate=False)

+------------+--------------+-----------+-----------+----------+------------+
|category    |total_quantity|total_lines|net_revenue|total_cost|total_margin|
+------------+--------------+-----------+-----------+----------+------------+
|Informatique|20            |10         |3724.00    |2800.00   |924.00      |
|Bureau      |30            |10         |5023.35    |3750.00   |1273.35     |
|Audio       |70            |20         |4512.45    |2600.00   |1912.45     |
+------------+--------------+-----------+-----------+----------+------------+



taux de marge par categorie

In [56]:
category_kpis = category_kpis.withColumn(
    "margin_rate_pct",
    F.when(
        F.col("net_revenue") != 0,
        F.round(
            F.col("total_margin") / F.col("net_revenue") * 100,
            2
        )
    )
)

category_kpis = category_kpis.orderBy(
    F.col("net_revenue").desc()
)

category_kpis.show(truncate=False)

+------------+--------------+-----------+-----------+----------+------------+---------------+
|category    |total_quantity|total_lines|net_revenue|total_cost|total_margin|margin_rate_pct|
+------------+--------------+-----------+-----------+----------+------------+---------------+
|Bureau      |30            |10         |5023.35    |3750.00   |1273.35     |25.35          |
|Audio       |70            |20         |4512.45    |2600.00   |1912.45     |42.38          |
|Informatique|20            |10         |3724.00    |2800.00   |924.00      |24.81          |
+------------+--------------+-----------+-----------+----------+------------+---------------+



In [57]:
# coherenfce verification of the KPIs
category_total = category_kpis.agg(
    F.sum("net_revenue").alias("total")
).first()["total"]

global_total = global_kpis.select(
    "net_revenue_total"
).first()[0]

assert category_total == global_total

print("Category reconciliation: OK")

Category reconciliation: OK


calculer le CA par produit

In [58]:
product_sales = (
    gold_lines
    .groupBy("product_id", "product_name", "category")
    .agg(
        F.sum("net_revenue").alias("total_revenue"),
        F.sum("quantity_parsed").alias("total_quantity")
    )
)

product_sales.orderBy(
    F.col("total_revenue").desc()
).show(truncate=False)

+----------+------------+------------+-------------+--------------+
|product_id|product_name|category    |total_revenue|total_quantity|
+----------+------------+------------+-------------+--------------+
|P008      |Printer     |Bureau      |5023.35      |30            |
|P003      |Screen      |Informatique|3724.00      |20            |
|P011      |Microphone  |Audio       |2419.95      |40            |
|P006      |Headphones  |Audio       |2092.50      |30            |
+----------+------------+------------+-------------+--------------+



window

In [59]:
from pyspark.sql.window import Window

revenue_window = Window.orderBy(
    F.col("total_revenue").desc()
)

row_window = Window.orderBy(
    F.col("total_revenue").desc(),
    F.col("product_id").asc()
)

calculer les trois rangs

In [60]:
product_ranking = (
    product_sales
    .withColumn(
        "rank",
        F.rank().over(revenue_window)
    )
    .withColumn(
        "dense_rank",
        F.dense_rank().over(revenue_window)
    )
    .withColumn(
        "row_number",
        F.row_number().over(row_window)
    )
)

product_ranking.orderBy("row_number").show(truncate=False)

+----------+------------+------------+-------------+--------------+----+----------+----------+
|product_id|product_name|category    |total_revenue|total_quantity|rank|dense_rank|row_number|
+----------+------------+------------+-------------+--------------+----+----------+----------+
|P008      |Printer     |Bureau      |5023.35      |30            |1   |1         |1         |
|P003      |Screen      |Informatique|3724.00      |20            |2   |2         |2         |
|P011      |Microphone  |Audio       |2419.95      |40            |3   |3         |3         |
|P006      |Headphones  |Audio       |2092.50      |30            |4   |4         |4         |
+----------+------------+------------+-------------+--------------+----+----------+----------+



classement par categorie

In [61]:
category_window = Window.partitionBy(
    "category"
).orderBy(
    F.col("total_revenue").desc()
)

product_ranking = product_ranking.withColumn(
    "category_rank",
    F.rank().over(category_window)
)

product_ranking.orderBy(
    "category", "category_rank", "product_id"
).show(truncate=False)

+----------+------------+------------+-------------+--------------+----+----------+----------+-------------+
|product_id|product_name|category    |total_revenue|total_quantity|rank|dense_rank|row_number|category_rank|
+----------+------------+------------+-------------+--------------+----+----------+----------+-------------+
|P011      |Microphone  |Audio       |2419.95      |40            |3   |3         |3         |1            |
|P006      |Headphones  |Audio       |2092.50      |30            |4   |4         |4         |2            |
|P008      |Printer     |Bureau      |5023.35      |30            |1   |1         |1         |1            |
|P003      |Screen      |Informatique|3724.00      |20            |2   |2         |2         |1            |
+----------+------------+------------+-------------+--------------+----+----------+----------+-------------+



In [62]:
# Persist Gold tables

gold_tables = {
    "sales_lines": gold_lines,
    "global_kpis": global_kpis,
    "category_kpis": category_kpis,
    "product_ranking": product_ranking
}

gold_dir = project_dir / "data_lakehouse" / "gold"

for name, df in gold_tables.items():
    path = gold_dir / name

    df.write.mode("overwrite").parquet(str(path))

    print(f"Saved: {name}")

Saved: sales_lines
Saved: global_kpis
Saved: category_kpis
Saved: product_ranking


In [63]:
for name, df in gold_tables.items():
    path = gold_dir / name

    reloaded = spark.read.parquet(str(path))

    assert reloaded.count() == df.count()

    print(f"{name}: persistence OK")

sales_lines: persistence OK
global_kpis: persistence OK
category_kpis: persistence OK
product_ranking: persistence OK


## SparkSQL

In [64]:
# Register Gold DataFrame as a SQL view
gold_lines.createOrReplaceTempView("gold_sales")

In [ ]:
# eligible commands and their dates
query_1 = """
SELECT DISTINCT
    order_id,
    order_date_parsed AS order_date
FROM gold_sales
ORDER BY order_id
"""

spark.sql(query_1).show(30, truncate=False)

+--------+----------+
|order_id|order_date|
+--------+----------+
|O0001   |2025-05-04|
|O0002   |2025-05-07|
|O0005   |2025-05-16|
|O0006   |2025-05-19|
|O0009   |2025-05-28|
|O0010   |2025-05-31|
|O0013   |2025-06-09|
|O0014   |2025-06-12|
|O0017   |2025-06-21|
|O0018   |2025-06-24|
|O0021   |2025-07-03|
|O0022   |2025-07-06|
|O0025   |2025-07-15|
|O0026   |2025-07-18|
|O0029   |2025-07-27|
|O0030   |2025-07-30|
|O0033   |2025-08-08|
|O0034   |2025-05-03|
|O0037   |2025-05-12|
|O0038   |2025-05-15|
+--------+----------+



In [66]:
# CA global
query_2 = """
SELECT
    SUM(net_revenue) AS total_revenue
FROM gold_sales
"""

spark.sql(query_2).show()

+-------------+
|total_revenue|
+-------------+
|     13259.80|
+-------------+



In [67]:
# Analyse par categorie
query_3 = """
SELECT
    category,
    SUM(net_revenue) AS total_revenue,
    SUM(total_cost) AS total_cost,
    SUM(margin) AS total_margin
FROM gold_sales
GROUP BY category
ORDER BY total_revenue DESC
"""

spark.sql(query_3).show(truncate=False)

+------------+-------------+----------+------------+
|category    |total_revenue|total_cost|total_margin|
+------------+-------------+----------+------------+
|Bureau      |5023.35      |3750.00   |1273.35     |
|Audio       |4512.45      |2600.00   |1912.45     |
|Informatique|3724.00      |2800.00   |924.00      |
+------------+-------------+----------+------------+



In [68]:
# Query 4: Identify Top-Performing Products by Revenue

query_4 = """
SELECT
    product_id,
    product_name,
    category,
    SUM(quantity_parsed) AS total_quantity,
    SUM(net_revenue) AS total_revenue,
    SUM(margin) AS total_margin
FROM gold_sales
GROUP BY product_id, product_name, category
ORDER BY total_revenue DESC, product_id
LIMIT 10
"""

spark.sql(query_4).show(truncate=False)

+----------+------------+------------+--------------+-------------+------------+
|product_id|product_name|category    |total_quantity|total_revenue|total_margin|
+----------+------------+------------+--------------+-------------+------------+
|P008      |Printer     |Bureau      |30            |5023.35      |1273.35     |
|P003      |Screen      |Informatique|20            |3724.00      |924.00      |
|P011      |Microphone  |Audio       |40            |2419.95      |1019.95     |
|P006      |Headphones  |Audio       |30            |2092.50      |892.50      |
+----------+------------+------------+--------------+-------------+------------+



In [69]:
# Query 5: Calculate Monthly Net Revenue

query_5 = """
SELECT
    DATE_FORMAT(order_date_parsed, 'yyyy-MM') AS sales_month,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(net_revenue) AS total_revenue
FROM gold_sales
GROUP BY DATE_FORMAT(order_date_parsed, 'yyyy-MM')
ORDER BY sales_month
"""

spark.sql(query_5).show(truncate=False)

+-----------+------------+-------------+
|sales_month|total_orders|total_revenue|
+-----------+------------+-------------+
|2025-05    |9           |5740.10      |
|2025-06    |4           |2754.95      |
|2025-07    |6           |4012.75      |
|2025-08    |1           |752.00       |
+-----------+------------+-------------+



In [70]:
# Query 6: Identify Top Customers by Generated Revenue

query_6 = """
SELECT
    customer_id,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(net_revenue) AS total_revenue,
    SUM(margin) AS total_margin
FROM gold_sales
GROUP BY customer_id
ORDER BY total_revenue DESC, customer_id
LIMIT 10
"""

spark.sql(query_6).show(truncate=False)

+-----------+------------+-------------+------------+
|customer_id|total_orders|total_revenue|total_margin|
+-----------+------------+-------------+------------+
|C001       |1           |940.00       |285.00      |
|C007       |1           |940.00       |285.00      |
|C023       |1           |940.00       |285.00      |
|C027       |1           |940.00       |285.00      |
|C029       |1           |940.00       |285.00      |
|C019       |1           |893.00       |238.00      |
|C003       |1           |867.35       |212.35      |
|C025       |1           |799.00       |144.00      |
|C021       |1           |752.00       |97.00       |
|C005       |1           |736.00       |81.00       |
+-----------+------------+-------------+------------+



In [71]:
# Query 7: Rank Products Within Each Category Using Window Functions

query_7 = """
WITH product_revenue AS (
    SELECT
        category,
        product_id,
        product_name,
        SUM(net_revenue) AS total_revenue
    FROM gold_sales
    GROUP BY category, product_id, product_name
)
SELECT
    category,
    product_id,
    product_name,
    total_revenue,
    RANK() OVER (
        PARTITION BY category
        ORDER BY total_revenue DESC
    ) AS rank,
    DENSE_RANK() OVER (
        PARTITION BY category
        ORDER BY total_revenue DESC
    ) AS dense_rank,
    ROW_NUMBER() OVER (
        PARTITION BY category
        ORDER BY total_revenue DESC, product_id
    ) AS row_number
FROM product_revenue
ORDER BY category, row_number
"""

spark.sql(query_7).show(truncate=False)

+------------+----------+------------+-------------+----+----------+----------+
|category    |product_id|product_name|total_revenue|rank|dense_rank|row_number|
+------------+----------+------------+-------------+----+----------+----------+
|Audio       |P011      |Microphone  |2419.95      |1   |1         |1         |
|Audio       |P006      |Headphones  |2092.50      |2   |2         |2         |
|Bureau      |P008      |Printer     |5023.35      |1   |1         |1         |
|Informatique|P003      |Screen      |3724.00      |1   |1         |1         |
+------------+----------+------------+-------------+----+----------+----------+



In [72]:
# Query 8: Compare Spark SQL Results with PySpark DataFrame Aggregations

sql_results = spark.sql("""
SELECT
    COUNT(DISTINCT order_id) AS total_orders,
    COUNT(*) AS total_lines,
    SUM(quantity_parsed) AS total_quantity,
    SUM(net_revenue) AS total_revenue,
    SUM(total_cost) AS total_cost,
    SUM(margin) AS total_margin
FROM gold_sales
""").first()

df_results = global_kpis.first()

assert sql_results["total_orders"] == df_results["eligible_orders"]
assert sql_results["total_lines"] == df_results["gold_lines_count"]
assert sql_results["total_quantity"] == df_results["total_quantity"]
assert sql_results["total_revenue"] == df_results["net_revenue_total"]
assert sql_results["total_cost"] == df_results["total_cost_total"]
assert sql_results["total_margin"] == df_results["total_margin"]

print("SQL vs PySpark DataFrame: ALL CHECKS PASSED")

SQL vs PySpark DataFrame: ALL CHECKS PASSED


### Validation

In [73]:
# Validation 1: Verify Financial Reconciliation

financial_totals = gold_lines.agg(
    F.sum("net_revenue").alias("revenue"),
    F.sum("total_cost").alias("cost"),
    F.sum("margin").alias("margin")
).first()

revenue = financial_totals["revenue"]
cost = financial_totals["cost"]
margin = financial_totals["margin"]

assert revenue - cost == margin

print("Net revenue:", revenue)
print("Total cost:", cost)
print("Total margin:", margin)
print("Financial reconciliation: PASSED")

Net revenue: 13259.80
Total cost: 9150.00
Total margin: 4109.80
Financial reconciliation: PASSED


In [74]:
# Validation 2: Compare Gold KPIs Against Reference Values

from decimal import Decimal

actual = global_kpis.first().asDict()

expected = {
    "eligible_orders": 20,
    "gold_lines_count": 40,
    "total_customers": 20,
    "total_quantity": 120,
    "net_revenue_total": Decimal("13259.80"),
    "total_cost_total": Decimal("9150.00"),
    "total_margin": Decimal("4109.80"),
    "margin_rate_pct": Decimal("30.99"),
    "average_order_value": Decimal("662.99")
}

mismatches = []

for key, reference in expected.items():
    observed = actual[key]
    passed = observed is not None and Decimal(str(observed)) == Decimal(str(reference))

    status = "PASS" if passed else "FAIL"
    print(f"{key:25} | Actual: {str(observed):12} | Expected: {reference} | {status}")

    if not passed:
        mismatches.append(key)

print("\nMismatches:", mismatches)

eligible_orders           | Actual: 20           | Expected: 20 | PASS
gold_lines_count          | Actual: 40           | Expected: 40 | PASS
total_customers           | Actual: 20           | Expected: 20 | PASS
total_quantity            | Actual: 120          | Expected: 120 | PASS
net_revenue_total         | Actual: 13259.80     | Expected: 13259.80 | PASS
total_cost_total          | Actual: 9150.00      | Expected: 9150.00 | PASS
total_margin              | Actual: 4109.80      | Expected: 4109.80 | PASS
margin_rate_pct           | Actual: 30.99        | Expected: 30.99 | PASS
average_order_value       | Actual: 662.99       | Expected: 662.99 | PASS

Mismatches: []


In [75]:
# Validation 3: Verify Persisted Gold Financial Totals

gold_path = project_dir / "data_lakehouse" / "gold" / "sales_lines"

gold_reloaded = spark.read.parquet(str(gold_path))

persisted_totals = gold_reloaded.agg(
    F.sum("net_revenue").alias("revenue"),
    F.sum("total_cost").alias("cost"),
    F.sum("margin").alias("margin")
).first()

assert persisted_totals["revenue"] == revenue
assert persisted_totals["cost"] == cost
assert persisted_totals["margin"] == margin
assert gold_reloaded.count() == gold_lines.count()

print("Persisted Gold reconciliation: PASSED")

Persisted Gold reconciliation: PASSED


#### Tests

In [76]:
# Test 1: Save Gold Baseline Before Restart

lake_dir = project_dir / "data_lakehouse"
gold_dir = lake_dir / "gold"
baseline_dir = lake_dir / "metadata" / "idempotence_baseline"

gold_table_names = [
    "sales_lines",
    "global_kpis",
    "category_kpis",
    "product_ranking"
]

for name in gold_table_names:
    df = spark.read.parquet(str(gold_dir / name))

    df.write.mode("overwrite").parquet(
        str(baseline_dir / name)
    )

    print(f"{name}: {df.count()} rows saved")

print("Gold baseline saved successfully")

sales_lines: 40 rows saved
global_kpis: 1 rows saved
category_kpis: 3 rows saved
product_ranking: 4 rows saved
Gold baseline saved successfully


In [77]:
# Test 2: Restart Spark Session

from pyspark.sql import SparkSession

spark.stop()

spark = (
    SparkSession.builder
    .appName("Eshop-Idempotence-Test")
    .master("local[*]")
    .getOrCreate()
)

print("New Spark session started")

New Spark session started


In [78]:
# Test 3: Validate Persisted Silver and Gold Tables

source_names = [
    "customers",
    "products",
    "promotions",
    "orders",
    "order_items"
]

for name in source_names:
    df = spark.read.parquet(
        str(lake_dir / "silver" / name)
    )
    print(f"Silver/{name}: {df.count()} rows")

print("-" * 40)

for name in gold_table_names:
    df = spark.read.parquet(
        str(gold_dir / name)
    )
    print(f"Gold/{name}: {df.count()} rows")

print("Persistent tables successfully reloaded")

Silver/customers: 30 rows
Silver/products: 12 rows
Silver/promotions: 5 rows
Silver/orders: 40 rows
Silver/order_items: 80 rows
----------------------------------------
Gold/sales_lines: 40 rows
Gold/global_kpis: 1 rows
Gold/category_kpis: 3 rows
Gold/product_ranking: 4 rows
Persistent tables successfully reloaded


In [79]:
# Test 4: Validate Pipeline Idempotence

for name in gold_table_names:
    before = spark.read.parquet(
        str(baseline_dir / name)
    )

    after = spark.read.parquet(
        str(gold_dir / name)
    )

    assert set(before.columns) == set(after.columns)

    cols = sorted(before.columns)

    before = before.select(*cols)
    after = after.select(*cols)

    assert before.count() == after.count(), (
        f"Row count mismatch: {name}"
    )

    assert before.exceptAll(after).count() == 0, (
        f"Changed or missing rows: {name}"
    )

    assert after.exceptAll(before).count() == 0, (
        f"New or duplicated rows: {name}"
    )

    print(f"{name}: IDEMPOTENT")

print("ALL IDEMPOTENCE TESTS PASSED")

sales_lines: IDEMPOTENT
global_kpis: IDEMPOTENT
category_kpis: IDEMPOTENT
product_ranking: IDEMPOTENT
ALL IDEMPOTENCE TESTS PASSED


In [80]:
# Final Check: Validate Persisted Gold KPIs

from decimal import Decimal

gold_dir = project_dir / "data_lakehouse" / "gold"

actual = spark.read.parquet(
    str(gold_dir / "global_kpis")
).first().asDict()

expected = {
    "eligible_orders": 20,
    "gold_lines_count": 40,
    "total_customers": 20,
    "total_quantity": 120,
    "net_revenue_total": Decimal("13259.80"),
    "total_cost_total": Decimal("9150.00"),
    "total_margin": Decimal("4109.80"),
    "margin_rate_pct": Decimal("30.99"),
    "average_order_value": Decimal("662.99")
}

failed = []

for key, target in expected.items():
    observed = actual[key]
    passed = Decimal(str(observed)) == Decimal(str(target))

    print(f"{key}: {'PASS' if passed else 'FAIL'}")
    if not passed:
        print(f"  Actual: {observed} | Expected: {target}")
        failed.append(key)

assert not failed, f"Failed KPI checks: {failed}"

print("\nALL GOLD KPI CHECKS PASSED")

eligible_orders: PASS
gold_lines_count: PASS
total_customers: PASS
total_quantity: PASS
net_revenue_total: PASS
total_cost_total: PASS
total_margin: PASS
margin_rate_pct: PASS
average_order_value: PASS

ALL GOLD KPI CHECKS PASSED


In [81]:
# Final Check: Validate Persisted Silver and Quarantine Counts

lake_dir = project_dir / "data_lakehouse"

expected_counts = {
    "customers": (30, 1),
    "products": (12, 1),
    "promotions": (5, 1),
    "orders": (40, 3),
    "order_items": (80, 5)
}

for name, (expected_valid, expected_rejected) in expected_counts.items():
    valid = spark.read.parquet(
        str(lake_dir / "silver" / name)
    ).count()

    rejected = spark.read.parquet(
        str(lake_dir / "quarantine" / name)
    ).count()

    assert valid == expected_valid, f"{name}: Silver mismatch"
    assert rejected == expected_rejected, f"{name}: Quarantine mismatch"

    print(f"{name}: PASS ({valid} valid, {rejected} rejected)")

print("\nALL SILVER CHECKS PASSED")

customers: PASS (30 valid, 1 rejected)
products: PASS (12 valid, 1 rejected)
promotions: PASS (5 valid, 1 rejected)
orders: PASS (40 valid, 3 rejected)
order_items: PASS (80 valid, 5 rejected)

ALL SILVER CHECKS PASSED
